In [39]:
import jax
import jax.numpy as jnp
import numpy as np

class DeepSeaExactValue:
    def __init__(self, size: int, unscaled_move_cost: float = 0.01, gamma: float = 0.99):
        self.N = size
        self.cost = unscaled_move_cost
        self.gamma = gamma
        self.num_grid_states = size * size
        self.num_total_states = self.num_grid_states + 1  # +1 for Absorbing Terminal
        self.terminal_idx = self.num_grid_states
        
        # 1. Pre-compute Observations Stack (N^2 x N x N)
        self.obs_stack = self._create_obs_stack()
        
        # 2. Pre-compute Transition Matrix P (M x A x M) 
        #    and Extrinsic Reward Matrix R (M x A)
        self.P, self.R_extrinsic = self._build_env_dynamics()

    def _create_obs_stack(self):
        """Creates a stack of one-hot observations for all grid states."""
        # Shape: (N^2, N, N, 1)
        obs_stack = np.zeros((self.num_grid_states, self.N, self.N), dtype=np.float32)
        idx = 0
        for r in range(self.N):
            for c in range(self.N):
                obs_stack[idx, r, c] = 1.0
                idx += 1
        return jnp.array(obs_stack)[...,None]

    def _build_env_dynamics(self):
            """
            Constructs the static P and R matrices.
            Actions: 0 = Left, 1 = Right
            """
            num_states = self.num_total_states
            num_actions = 2
            
            P = np.zeros((num_states, num_actions, num_states))
            R = np.zeros((num_states, num_actions))
            
            curr_idx = 0 # Initialize to ensure scope safety if N=0 (edge case)

            # Loop over all non-terminal grid states
            for r in range(self.N):
                for c in range(self.N):
                    curr_idx = r * self.N + c
                    
                    # --- Action 0: Left ---
                    # Transition: r -> r+1, c -> max(0, c-1)
                    next_r = r + 1
                    next_c = max(0, c - 1)
                    
                    if next_r >= self.N:
                        next_idx = self.terminal_idx
                    else:
                        next_idx = next_r * self.N + next_c
                    
                    P[curr_idx, 0, next_idx] = 1.0
                    R[curr_idx, 0] = 0.0  # Zero cost for Left

                    # --- Action 1: Right ---
                    # Transition: r -> r+1, c -> c+1
                    next_r_right = r + 1
                    next_c_right = c + 1 
                    
                    if next_c_right >= self.N:
                        next_c_right = self.N - 1
                    
                    if next_r_right >= self.N:
                        next_idx_right = self.terminal_idx
                    else:
                        next_idx_right = next_r_right * self.N + next_c_right

                    P[curr_idx, 1, next_idx_right] = 1.0
                    
                    # --- Standard Move Cost ---
                    # Apply the cost to ALL right moves initially
                    R[curr_idx, 1] = -(self.cost / self.N)
                    

            # --- Goal Reward Override ---
            # The loop finishes with curr_idx = (N-1)*N + (N-1), which is the Bottom-Right state.
            R[curr_idx, 1] += 1.0 

            # --- Terminal State Handling ---
            # Absorbing state: transitions to itself, 0 reward
            P[self.terminal_idx, :, self.terminal_idx] = 1.0
            R[self.terminal_idx, :] = 0.0
            
            return jnp.array(P), jnp.array(R)

    def solve_true_value(self, pi_grid: jax.Array, R_matrix: jax.Array):
            """
            Computes V_pi = (I - gamma * P_pi)^-1 * R_pi using a tabular policy.

            Args:
                pi_grid: Shape (N^2, 2). The policy probabilities for the grid states.
                R_custom: Optional Shape (M, 2). If provided, computes value w.r.t this reward
                        matrix instead of self.R_extrinsic. Useful for intrinsic rewards.
            """

            # 2. Append dummy policy for the terminal state
            # The terminal state absorbs, so policy doesn't strictly matter, 
            # but we need dimensions to match (M x 2).
            terminal_policy = jnp.array([[1.0, 0.0]]) 
            pi = jnp.vstack([pi_grid, terminal_policy])
            
            # 3. Compute P^pi (Transition matrix induced by policy)
            # P^pi[s, s'] = sum_a (pi[s, a] * P[s, a, s'])
            # Shape: (M, M)
            P_pi = jnp.einsum('sa, sam -> sm', pi, self.P)
            
            # 4. Compute R^pi (Expected reward vector induced by policy)
            # R^pi[s] = sum_a (pi[s, a] * R[s, a])
            # Shape: (M,)
            R_pi = jnp.einsum('sa, sa -> s', pi, R_matrix)
            
            # 5. Solve Linear System: (I - gamma * P^pi) V = R^pi
            I = jnp.eye(self.num_total_states)
            A = I - self.gamma * P_pi
            
            V_true = jnp.linalg.solve(A, R_pi)
            
            return V_true
    
    def get_value_grid(self, V_flat: jax.Array) -> jax.Array:
            """
            Reshapes the flat value vector (N^2 + 1) into an N x N grid.
            Discards the terminal state value.
            """
            # Take the first N^2 elements
            V_grid_flat = V_flat[:self.num_grid_states]
            
            # Reshape to (N, N)
            return V_grid_flat.reshape((self.N, self.N))

In [40]:
N = 5
evaluator = DeepSeaExactValue(size=N, unscaled_move_cost=0.01)
evaluator.obs_stack.shape

(25, 5, 5, 1)

In [ ]:
pi_grid = jnp.ones((evaluator.num_grid_states,2)) / 2
V = evaluator.solve_true_value(pi_grid, evaluator.R_extrinsic)
evaluator.get_value_grid(V)

Array([[ 0.02511763,  0.02511763,  0.11517351,  0.11517351,  0.17521077],
       [-0.0039404 ,  0.05670329,  0.05670329,  0.17799067,  0.17799067],
       [-0.0029701 , -0.0029701 ,  0.11954241,  0.11954241,  0.24205491],
       [-0.00199   , -0.00199   , -0.00199   ,  0.24551001,  0.24551001],
       [-0.001     , -0.001     , -0.001     , -0.001     ,  0.499     ]],      dtype=float32)

In [44]:
import networks
rng = jax.random.PRNGKey(0)
config = {}
config["NETWORK_TYPE"] = 'cnn'
config['BONUS_SCALE'] = 1.96
obs_shape = (N,N,1)
n_actions=2
network, network_params = networks.initialize_actor_critic(rng, obs_shape, n_actions, config, n_heads=2)
pi, v = network.apply(network_params, evaluator.obs_stack)

In [46]:
V_flat = evaluator.solve_true_value(pi.probs, evaluator.R_extrinsic)

# 2. Convert to Grid
V_grid = evaluator.get_value_grid(V_flat)

print("Value Grid Shape:", V_grid.shape)
print(V_grid)

Value Grid Shape: (5, 5)
[[ 0.02518906  0.02519639  0.11534746  0.11534128  0.1753282 ]
 [-0.00394365  0.05684052  0.05678929  0.17816527  0.17805381]
 [-0.00297158 -0.00297439  0.11965877  0.11956666  0.24213548]
 [-0.00199152 -0.0019914  -0.00199174  0.24570444  0.24547812]
 [-0.00100009 -0.00100076 -0.00100006 -0.00100012  0.49893746]]


In [47]:
rnd_net, rnd_params = networks.initialize_rnd_network(rng, obs_shape, config)
_, target_params = networks.initialize_rnd_network(rng, obs_shape, config)
get_features_fn = lambda obs: rnd_net.apply(target_params, obs)
batch_get_features = jax.vmap(get_features_fn)
batch_get_features(evaluator.obs_stack).shape

S = jnp.eye(128)
N = 1000

def get_int_rew(S, features, N):
    Sigma_inv = jnp.linalg.solve(S, jnp.eye(features.shape[-1]))
    bonus_sq = jnp.einsum('...i,ij,...j->...', features, Sigma_inv, features) / jnp.maximum(1.0, N)
    rho = config['BONUS_SCALE'] * jnp.sqrt(jnp.maximum(bonus_sq, 0.0))
    return rho

r_next_grid = get_int_rew(S, batch_get_features(evaluator.obs_stack), N) # N^2
# add terminal state
r_next_all = jnp.concatenate([r_next_grid, jnp.array([0.0])]) # N^2 + 1
# R(s,a) =sum_{s'} R(s') P(s' | s, a)
R_int_sa = jnp.einsum('sam, m -> sa', evaluator.P, r_next_all) # (N^2 + 1, A)

In [48]:
r_next_grid

Array([0.04902839, 0.04844978, 0.04531921, 0.05073614, 0.03548877,
       0.051465  , 0.09733525, 0.05506155, 0.08470191, 0.05518308,
       0.04271216, 0.05884335, 0.04551725, 0.0536109 , 0.04441729,
       0.05611018, 0.09272838, 0.04883509, 0.08025651, 0.05112778,
       0.04525717, 0.058491  , 0.04570626, 0.05273889, 0.0410101 ],      dtype=float32)

In [49]:
V_flat_i = evaluator.solve_true_value(pi.probs, R_int_sa)

# 2. Convert to Grid
V_grid_i = evaluator.get_value_grid(V_flat_i)

print("Value Grid Shape:", V_grid_i.shape)
print(V_grid_i)

Value Grid Shape: (5, 5)
[[0.23932803 0.21273345 0.25612688 0.20949689 0.22717674]
 [0.16378179 0.16939676 0.15838245 0.16416132 0.15349293]
 [0.12261461 0.10567664 0.13046603 0.10071079 0.11035748]
 [0.05188131 0.04548201 0.0556112  0.04335603 0.04687452]
 [0.         0.         0.         0.         0.        ]]


In [58]:
from utils import load_run_data
import matplotlib.pyplot as plt
config, metrics = load_run_data("count_rew_prop/20260107_134635", "DeepSea-bsuite", 'results')
metrics.keys()

dict_keys(['bonus_max', 'bonus_mean', 'bonus_std', 'discount', 'feat_norm', 'i_val_const_obs', 'intrinsic_rew_mean', 'intrinsic_rew_std', 'intrinsic_v_mean', 'intrinsic_v_std', 'lambda_ret_mean', 'lambda_ret_std', 'mean_rew', 'ppo_loss', 'returned_discounted_episode_returns', 'returned_episode', 'returned_episode_lengths', 'returned_episode_returns', 'rnd_loss', 'v_e', 'v_e_pred', 'v_i', 'v_i_pred'])

In [62]:
metrics['v_i'][0][10]

Array([[ 0.9499216 ,  0.9457861 ,  0.9483825 ,  0.9483853 ,  0.948386  ],
       [-0.00785771,  0.9620469 ,  0.95782024,  0.95998055,  0.9599852 ],
       [-0.00593257, -0.00592756,  0.9738097 ,  0.97146827,  0.9716974 ],
       [-0.00397025, -0.00397642, -0.00396918,  0.9858207 ,  0.9835313 ],
       [-0.00199786, -0.00199667, -0.00199708, -0.0019985 ,  0.9978216 ]],      dtype=float32)

In [61]:
metrics['v_e'][0][10]

Array([[0.04848   , 1.1031157 , 1.0584165 , 0.95341885, 0.95337987],
       [0.11493032, 0.03725708, 0.69278044, 0.57518274, 0.5755153 ],
       [0.05693827, 0.08398582, 0.02542205, 0.33850336, 0.3384538 ],
       [0.03945373, 0.02960187, 0.04593671, 0.01289955, 0.01297657],
       [0.        , 0.        , 0.        , 0.        , 0.        ]],      dtype=float32)

In [63]:
metrics['v_i_pred'][0][10]

Array([[1.05813384e-01, 1.19155094e-01, 2.03994997e-02, 8.18586275e-02,
        5.38735688e-02],
       [1.45769715e-01, 7.25786835e-02, 5.49309105e-02, 6.71769381e-02,
        5.04144430e-02],
       [8.94629657e-02, 1.11272871e-01, 4.73061539e-02, 5.84357753e-02,
        2.96892226e-02],
       [5.23306951e-02, 4.36347611e-02, 6.30786493e-02, 2.31299605e-02,
        3.11368592e-02],
       [1.94658060e-04, 4.83178766e-04, 6.63007144e-04, 5.05624805e-04,
        1.40052754e-04]], dtype=float32)

In [64]:
metrics['v_e_pred'][0][10]

Array([[0.9440001 , 0.46126372, 0.5184719 , 0.24862449, 0.6421673 ],
       [0.05719679, 0.9593319 , 0.63217777, 0.66875666, 0.41276848],
       [0.0821941 , 0.0094041 , 0.9715748 , 0.41202188, 0.40213883],
       [0.11569104, 0.01131925, 0.02402236, 0.98566616, 0.668821  ],
       [0.16940863, 0.07368968, 0.01719008, 0.00220683, 0.9982655 ]],      dtype=float32)